# Reasoning agent

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "submission":
    os.chdir("..")
for name, folder in {"HF_HOME": "huggingface", "XDG_CACHE_HOME": "libraries",
                     "TORCH_HOME": "torch", "TRITON_CACHE_DIR": "triton",
                     "CUDA_CACHE_PATH": "cuda"}.items():
    os.environ[name] = str((Path("data/cache") / folder).resolve())

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'

## Representation

In [2]:
import itertools
import json
import re
import time
from functools import lru_cache
from graphlib import TopologicalSorter
from pathlib import Path

import torch
import xgrammar as xgr
import z3
from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer
from z3.z3util import get_vars
from xgrammar.contrib.hf import LogitsProcessor


MODELS = {
    "Qwen/Qwen3-4B-Instruct-2507": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "Qwen/Qwen3.5-4B": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    "Qwen/Qwen3.5-9B": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",
}

LOGIC_PROMPT = """Translate the input into first-order logic, without answering or solving the query.
Return only JSON with two keys: premises (one formula per input premise, in order) and query.
Each formula is a JSON array:
  ["predicate", "constant", ...] for an atomic property or relation;
  ["not", formula], ["and", formula, ...], ["or", formula, ...],
  ["implies", antecedent, consequent], ["iff", formula, formula], ["xor", formula, formula];
  ["forall", "?x", formula] or ["exists", "?x", formula];
  ["=", "term", "term"] for identity.
Variables begin with ?. Free variables in premises are implicitly universal;
use exists explicitly for existential statements. Constants are ordinary strings.
Use the same predicate and constant for the same meaning throughout. Nest quantifiers for
multiple variables. Preserve negation scope, quantifiers, relations and only-if direction.
Do not add inferred facts, outside knowledge or type memberships. A rule does not assert its
antecedent. The query must be a formula, never a true/false label. No declarations are needed.

Input: {"premises":["All cats are animals.","Milo is a cat."],"query":"Milo is an animal."}
Output: {"premises":[["forall","?x",["implies",["cat","?x"],["animal","?x"]]],["cat","milo"]],"query":["animal","milo"]}

Input: {"premises":["Some artists are not musicians."],"query":"Every artist is a musician."}
Output: {"premises":[["exists","?x",["and",["artist","?x"],["not",["musician","?x"]]]]],"query":["forall","?x",["implies",["artist","?x"],["musician","?x"]]]}
"""

CAUSAL_PROMPT = """Extract a Boolean structural causal model from the input; do not answer
the query or compute any event values. Return only JSON with these keys:
variables: list of all event names; equations: mapping of each dependent event to its
Boolean defining expression; observations: list of factual Boolean expressions;
interventions: mapping of intervened event names to JSON booleans;
priors: mapping of events with Bernoulli priors to their probabilities;
query: the queried event, or its negation, as a Boolean expression.
An expression is an event name string, or a JSON array ["and", "a", "b"],
["or", "a", "b"], ["not", "a"]. Use the same lowercase names everywhere.
'A causes not B' gives B = ["not", "a"]. Multiple effects each get their own equation.
Observations describe the factual world, not interventions. 'Instead of X' supplies
factual observation X. Collect both joint and sequential interventions, without propagating
their effects yourself. Unmentioned equations, observations and priors stay absent.
Example: 'A causes B. Would B occur if not A instead of A?'
{"variables":["a","b"],"equations":{"b":"a"},"observations":["a"],
"interventions":{"a":false},"priors":{},"query":"b"}
"""

OPERATORS = {"and": z3.And, "or": z3.Or, "not": z3.Not, "implies": z3.Implies,
             "xor": z3.Xor, "iff": lambda a, b: a == b}

## Constrained decoding

In [3]:
def output_schema(mode, count):
    ref = {"$ref": "#/$defs/node"}
    string = {"type": "string"}

    def fixed(*items):
        return {"type": "array", "prefixItems": list(items), "items": False,
                "minItems": len(items), "maxItems": len(items)}

    if mode == "causal":
        expression = {"anyOf": [string, ref]}
        node = {"anyOf": [fixed({"enum": ["not"]}, expression),
                          {"type": "array", "prefixItems": [{"enum": ["and", "or"]}],
                           "items": expression, "minItems": 3}]}
        properties = {
            "variables": {"type": "array", "items": {"type": "string"}},
            "equations": {"type": "object", "additionalProperties": expression},
            "observations": {"type": "array", "items": expression},
            "interventions": {"type": "object", "additionalProperties": {"type": "boolean"}},
            "priors": {"type": "object", "additionalProperties": {"type": "number"}},
            "query": expression,
        }
    else:
        node = {"anyOf": [
            {"type": "array", "items": string, "minItems": 1},
            fixed({"enum": ["not"]}, ref),
            fixed({"enum": ["implies", "iff", "xor"]}, ref, ref),
            {"type": "array", "prefixItems": [{"enum": ["and", "or"]}], "items": ref, "minItems": 2},
            fixed({"enum": ["forall", "exists"]}, string, ref),
            fixed({"enum": ["="]}, string, string),
        ]}
        properties = {"premises": {"type": "array", "items": ref, "minItems": count, "maxItems": count},
                      "query": ref}
    return {"type": "object", "properties": properties, "required": list(properties),
            "additionalProperties": False, "$defs": {"node": node}}


@lru_cache(maxsize=2)
def grammar_compiler(tokenizer, vocab_size):
    return xgr.GrammarCompiler(xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=vocab_size))

## Semantic parser

In [4]:
def load_model(repo, data=Path("data")):
    from huggingface_hub import snapshot_download

    path = data / "models" / repo.split("/")[-1]
    if not path.exists():
        snapshot_download(repo, revision=MODELS[repo], local_dir=path,
                          cache_dir=data / "cache/huggingface/hub",
                          allow_patterns=["*.json", "*.safetensors", "*.jinja", "*.txt"])
    config = AutoConfig.from_pretrained(path)
    loader = AutoModelForImageTextToText if hasattr(config, "vision_config") else AutoModelForCausalLM
    device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device != "cpu" else torch.float32
    model = loader.from_pretrained(path, dtype=dtype).to(device).eval()
    model.generation_config.update(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
    tokenizer = AutoTokenizer.from_pretrained(path, padding_side="left")
    tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def sentences(context):
    return re.split(r"(?<=[.!?])\s+|\n+", context.strip()) if isinstance(context, str) else context


def translate(rows, model, tokenizer, mode="classical"):
    prompt = CAUSAL_PROMPT if mode == "causal" else LOGIC_PROMPT
    if mode == "cwa":
        prompt += "\nIn this rule language, someone, something, people and things range over all entities; do not introduce a person or thing type predicate. Translate explicit negatives only; do not complete missing information."
    dialogs = []
    for row in rows:
        payload = json.dumps({"premises": sentences(row["context"]), "query": row["statement"]})
        dialog = [{"role": "system", "content": prompt}, {"role": "user", "content": payload}]
        dialogs.append(dialog)
    texts = [tokenizer.apply_chat_template(d, tokenize=False, add_generation_prompt=True,
                                          enable_thinking=False) for d in dialogs]
    inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
    compiler = grammar_compiler(tokenizer, model.get_output_embeddings().weight.shape[0])
    grammars = [compiler.compile_json_schema(json.dumps(output_schema(mode, len(sentences(r["context"]))))) for r in rows]
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=2048,
                                logits_processor=[LogitsProcessor(grammars)],
                                pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

## First-order logic

In [5]:
def parse_logic(data):
    if set(data) != {"premises", "query"}:
        raise ValueError("Expected only premises and query, without an answer field")
    entity = z3.DeclareSort("Entity")
    constants, predicates, free_variables = {}, {}, {}

    def term(name, variables):
        if not isinstance(name, str):
            raise ValueError("Logical terms must be strings")
        if name in variables:
            return variables[name]
        if name.startswith("?"):
            return free_variables.setdefault(name, z3.FreshConst(entity))
        return constants.setdefault(name, z3.Const(name, entity))

    def formula(tree, variables):
        if not isinstance(tree, list) or not tree:
            raise ValueError("Expected a logical formula, not a supplied truth label")
        op, *args = tree
        if not isinstance(op, str):
            raise ValueError("Expected a predicate or logical operator")
        if op in ("forall", "exists"):
            name, body = args
            var = z3.FreshConst(entity)
            quantifier = z3.ForAll if op == "forall" else z3.Exists
            return quantifier([var], formula(body, {**variables, name: var}))
        if op in OPERATORS:
            return OPERATORS[op](*(formula(a, variables) for a in args))
        if op == "=":
            return term(args[0], variables) == term(args[1], variables)
        if op.lower() in {"true", "false", "yes", "no", "unknown", "uncertain"}:
            raise ValueError("A truth label is not a predicate")
        if op in predicates and predicates[op].arity() != len(args):
            raise ValueError(f"Predicate {op} has {len(args)} arguments; expected {predicates[op].arity()}")
        predicate = predicates.setdefault(op, z3.Function(op, *([entity] * len(args)), z3.BoolSort()))
        return predicate(*(term(a, variables) for a in args))

    premises = []
    for tree in data["premises"]:
        free_variables.clear()
        f = formula(tree, {})
        premises.append(z3.ForAll(list(free_variables.values()), f) if free_variables else f)
    free_variables.clear()
    query = formula(data["query"], {})
    if free_variables:
        raise ValueError("Query variables must be explicitly quantified")
    if z3.is_true(query) or z3.is_false(query):
        raise ValueError("The query must not supply its own truth value")
    return premises, query, list(constants.values())

## Entailment

In [6]:
def solve(premises, query):
    solver = z3.Solver()
    solver.set(timeout=2000)
    solver.add(*premises)
    consistency = solver.check()
    if consistency != z3.sat:
        return {"flag": consistency == z3.unsat, "status": "inconsistent" if consistency == z3.unsat else "timeout"}
    solver.push()
    solver.add(z3.Not(query))
    verdict = solver.check()
    solver.pop()
    if verdict == z3.unsat:
        return {"flag": True, "status": "entailed"}
    if verdict == z3.unknown:
        return {"flag": False, "status": "timeout"}
    solver.add(query)
    opposite = solver.check()
    return {"flag": False, "status": "refuted" if opposite == z3.unsat else "unknown" if opposite == z3.sat else "timeout"}


def closed_world(premises, query, constants):
    facts, rules = set(), []

    def conjunction(f):
        return [a for child in f.children() for a in conjunction(child)] if z3.is_and(f) else [f]

    def atom_key(f):
        if not (z3.is_app(f) and z3.is_bool(f) and f.decl().kind() == z3.Z3_OP_UNINTERPRETED):
            raise ValueError("Expected an atomic predicate in the rule program")
        return str(f)

    def add(f):
        if z3.is_quantifier(f):
            if not f.is_forall():
                raise ValueError("Closed-world rules must be universally quantified")
            for values in itertools.product(constants, repeat=f.num_vars()):
                add(z3.substitute_vars(f.body(), *values))
        elif z3.is_and(f):
            for child in f.children():
                add(child)
        elif z3.is_implies(f):
            body = conjunction(f.arg(0))
            positive = {atom_key(a) for a in body if not z3.is_not(a)}
            negative = {atom_key(a.arg(0)) for a in body if z3.is_not(a)}
            rules.extend((positive, negative, atom_key(h)) for h in conjunction(f.arg(1)) if not z3.is_not(h))
        elif z3.is_not(f):
            # Explicit negatives do not change negation-as-failure support.
            atom_key(f.arg(0))
        else:
            facts.add(atom_key(f))

    for premise in premises:
        add(premise)

    def closure(assumed):
        result = set(facts)
        while True:
            new = {head for positive, negative, head in rules
                   if positive <= result and not negative & assumed} - result
            if not new:
                return result
            result.update(new)

    lower = set()
    upper = facts | {a for p, n, h in rules for a in p | n | {h}}
    while True:
        new_lower = closure(upper)
        new_upper = closure(new_lower)
        if (lower, upper) == (new_lower, new_upper):
            break
        lower, upper = new_lower, new_upper
    flag = atom_key(query.arg(0)) not in upper if z3.is_not(query) else atom_key(query) in lower
    return {"flag": flag, "status": "fixed_point"}

## Causal inference

In [7]:
def causal(data):
    if set(data) != {"variables", "equations", "observations", "interventions", "priors", "query"}:
        raise ValueError("Expected a causal model, without an answer field")
    if any(not 0 <= p <= 1 for p in data["priors"].values()):
        raise ValueError("Prior probabilities must be between zero and one")
    names = set(data["variables"]) | data["interventions"].keys()
    variables = {name: z3.Bool(name) for name in names}
    changed = {name: z3.FreshBool(name + "_cf") for name in names}

    def expression(text):
        if isinstance(text, str):
            return variables[text]
        return OPERATORS[text[0]](*(expression(a) for a in text[1:]))

    equations = {name: expression(term) for name, term in data["equations"].items()}
    list(TopologicalSorter({name: {str(v) for v in get_vars(term)} for name, term in equations.items()}).static_order())
    substitutions = list(zip(variables.values(), changed.values()))
    factual = [variables[name] == term for name, term in equations.items()]
    factual.extend(expression(term) for term in data["observations"])
    for name in names - equations.keys():
        prior = data["priors"].get(name, 1.0)
        if prior in (0, 1):
            factual.append(variables[name] == bool(prior))
    counterfactual = []
    for name in names:
        if name in data["interventions"]:
            value = z3.BoolVal(data["interventions"][name])
        else:
            value = z3.substitute(equations[name], *substitutions) if name in equations else variables[name]
        counterfactual.append(changed[name] == value)
    query = z3.substitute(expression(data["query"]), *substitutions)
    consistency = z3.Solver()
    consistency.add(*factual)
    if consistency.check() == z3.unsat:
        return {"flag": False, "status": "inconsistent_evidence"}
    return solve([*factual, *counterfactual], query)


def reason(text, mode="classical", premise_count=None):
    if mode not in {"classical", "cwa", "causal"}:
        raise ValueError(f"Unknown reasoning mode: {mode}")
    text = text.strip()
    if text.startswith("```"):
        text = "\n".join(text.splitlines()[1:-1])
    if mode == "causal":
        return causal(json.loads(text))
    premises, query, constants = parse_logic(json.loads(text))
    if premise_count is not None and len(premises) != premise_count:
        raise ValueError(f"Expected {premise_count} premise assertions, found {len(premises)}")
    return closed_world(premises, query, constants) if mode == "cwa" else solve(premises, query)


def score(row, text):
    begin = time.perf_counter()
    try:
        prediction = reason(text, row["mode"], len(sentences(row["context"])))
        error = None
    except (ValueError, TypeError, KeyError, IndexError, z3.Z3Exception) as e:
        prediction = {"flag": False, "status": "parse_error"}
        error = str(e)
    return {**row, **prediction, "translation": text, "error": error,
            "reasoning_seconds": time.perf_counter() - begin}

## Rule grammar

In [8]:
import json
import re

from lark import Lark, Tree
from lark.exceptions import LarkError
from lemminflect import getLemma


RULE_GRAMMAR = r"""
?start: "If"i clauses [","] "then" clauses -> rule
      | ["All"i] adjectives ("people"i|"things"i) "are"i adjectives -> universal
      | clauses
clauses: clause ("and"i (clause|adjective))*
?clause: subject ("is"i|"are"i) adjectives -> property
       | subject ("does"i|"do"i) "not"i WORD entity -> negative_relation
       | subject WORD entity -> relation
adjectives: adjective (("," ["and"i]|"and"i) adjective)*
adjective: ["not"i] WORD
?subject: "someone"i -> variable
        | "something"i -> variable
        | "they"i -> variable
        | "it"i -> variable
        | entity
?entity: NAME -> name
       | "the"i WORD+ -> name
NAME: /[A-Z][\w-]*/
WORD: /[a-z]+/i
%import common.WS
%ignore WS
"""

RULE_PARSER = Lark(RULE_GRAMMAR, parser="earley", maybe_placeholders=False,
                   keep_all_tokens=True)


def rule_translation(context, statement, mode="cwa"):
    if isinstance(context, list):
        context = ". ".join(s.rstrip(". ") for s in context)
    def conjunction(items):
        return items[0] if len(items) == 1 else ["and", *items]

    def convert(tree):
        kind, children = tree.data, tree.children
        nodes = [c for c in children if isinstance(c, Tree)]
        words = [str(c).lower() for c in children if not isinstance(c, Tree)]
        if kind == "name":
            return "_".join(w for w in words if w != "the")
        if kind == "variable":
            return "?x"
        if kind == "adjective":
            return words[-1], "not" in words
        if kind == "adjectives":
            return [convert(n) for n in nodes]
        if kind == "property":
            subject = convert(nodes[0])
            literals = [[word, subject] if not negative else ["not", [word, subject]]
                        for word, negative in convert(nodes[1])]
            return subject, conjunction(literals)
        if kind in {"relation", "negative_relation"}:
            subject, obj = map(convert, nodes)
            verb = next(w for w in words if w not in {"does", "do", "not"})
            verb = getLemma(verb, upos="VERB")[0]
            atom = [verb, subject, obj]
            return subject, ["not", atom] if kind == "negative_relation" else atom
        if kind == "clauses":
            formulas, subject = [], None
            for node in nodes:
                if node.data == "adjective":
                    word, negative = convert(node)
                    atom = [word, subject]
                    formulas.append(["not", atom] if negative else atom)
                else:
                    subject, formula = convert(node)
                    formulas.append(formula)
            return conjunction(formulas)
        if kind == "rule":
            return ["implies", *map(convert, nodes)]
        if kind == "universal":
            sides = [conjunction([["not", [word, "?x"]] if negative else [word, "?x"]
                                  for word, negative in convert(n)]) for n in nodes]
            return ["implies", *sides]
        raise ValueError(f"Unsupported rule form: {kind}")

    premises = []
    for sentence in re.split(r"\.\s*", context):
        if not sentence.strip():
            continue
        tree = RULE_PARSER.parse(sentence.strip())
        if mode == "classical" and tree.data == "universal" and re.search(r"\bpeople\b", sentence, re.I):
            raise ValueError("A restricted quantifier requires semantic parsing")
        formula = convert(tree)
        if tree.data not in {"rule", "universal"} and "?x" in json.dumps(formula):
            raise ValueError("A pronoun in a fact requires semantic parsing")
        premises.append(formula)
    query = convert(RULE_PARSER.parse(statement.rstrip(". ")))
    return json.dumps({"premises": premises, "query": query})

## Causal grammar

In [9]:
def causal_translation(context, statement):
    def literal(text):
        text = re.sub(r"[.?,]", "", text).strip().lower()
        text = re.sub(r"^(\w+) not occurs?$", r"not \1", text)
        text = re.sub(r"\s+occurs?$", "", text)
        match = re.fullmatch(r"(not\s+)?([\w-]+)", text)
        if not match:
            raise ValueError(f"Unsupported event: {text}")
        return match[2], not bool(match[1])

    def expression(text):
        text = re.sub(r"\btogether\b", "", text, flags=re.I).strip()
        for word in ["or", "and"]:
            parts = re.split(rf"\s+{word}\s+", text, flags=re.I)
            if len(parts) > 1:
                return [word, *map(expression, parts)]
        name, positive = literal(text)
        return name if positive else ["not", name]

    text = " ".join(context) if isinstance(context, list) else context
    graph = re.split(r"\.(?!\d)|\bWe observed\b|\b\w+\s*~", text, maxsplit=1, flags=re.I)[0]
    graph = re.sub(r"^We know that\s+", "", graph, flags=re.I)
    equations = {}
    for clause in re.split(r",\s*(?:and\s+)?(?=[^,]*\bcauses?\b)", graph, flags=re.I):
        match = re.fullmatch(r"\s*(.+?)\s+causes?\s+(.+?)\s*", clause, re.I)
        if not match:
            raise ValueError(f"Unsupported causal clause: {clause}")
        cause = expression(match[1])
        for effect in re.split(r",\s*(?:and\s+)?|\s+and\s+", match[2], flags=re.I):
            name, positive = literal(effect)
            term = cause if positive else ["not", cause]
            equations[name] = ["or", equations[name], term] if name in equations else term
    priors = {name.lower(): float(p) for name, p in re.findall(r"(\w+)\s*~\s*Bern\(([0-9.]+)\)", text, re.I)}
    observations = []
    for obs in re.findall(r"We observed\s+([^.]*)", text, re.I):
        observations.extend(expression(p) for p in re.split(r",\s*(?:and\s+)?|\s+and\s+", obs))
    target = re.search(r"Would\s+(.+?)\s+occur", statement, re.I)
    if not target:
        raise ValueError("Missing causal target")
    if re.match(r"^Assume\b", statement, re.I):
        interventions = re.findall(r"(?:Assume|suppose)\s+(.+?)(?=,|\.|$)", statement, re.I)
    else:
        match = re.search(r"\bif\s+(.+)$", statement, re.I)
        if not match:
            raise ValueError("Missing intervention")
        interventions = [match[1].split(" instead of ")[0]]
        factual = re.search(r"\binstead of\s+(.+)$", statement, re.I)
        if factual:
            observations.extend(expression(p) for p in re.split(r"\s+and\s+", factual[1], flags=re.I))
    forced = dict(literal(p) for item in interventions for p in re.split(r"\s+and\s+", item, flags=re.I))
    def names(term):
        return {term} if isinstance(term, str) else set().union(*(names(t) for t in term[1:]))

    query = expression(target[1])
    variables = set(equations) | set(priors) | set().union(*(names(t) for t in equations.values()))
    if not names(query).union(*(names(t) for t in observations)) <= variables:
        raise ValueError("Unknown event in query or observation")
    return json.dumps({"variables": sorted(variables | forced.keys()), "equations": equations,
                       "observations": observations, "interventions": forced, "priors": priors,
                       "query": query})


def parse_controlled(context, statement, mode):
    try:
        return causal_translation(context, statement) if mode == "causal" else rule_translation(context, statement, mode)
    except (ValueError, TypeError, LarkError):
        return None

## Logic input

In [10]:
import json
import re

from lark import Lark, Transformer


LOGIC_GRAMMAR = r"""
    ?start: equivalence
    ?equivalence: implication (IFF implication)*   -> equivalence
    ?implication: disjunction (IMPLIES implication)? -> implication
    ?disjunction: exclusive (OR exclusive)*         -> disjunction
    ?exclusive: conjunction (XOR conjunction)*     -> exclusive
    ?conjunction: unary (AND unary)*                 -> conjunction
    ?unary: NOT unary                                -> negation
          | QUANT names [DOT] equivalence            -> quantified
          | "(" equivalence ")"
          | atom
    ?atom: NAME "(" [terms] ")"                     -> predicate
         | NAME EQUAL NAME                           -> equality
         | NAME                                      -> proposition
    names: NAME ("," NAME)*
    terms: NAME ("," NAME)*
    QUANT.2: /∀|∃|(?i:forall|exists)\b/
    NOT.2: /¬|~|!|(?i:not)\b/
    AND.2: /∧|&|(?i:and)\b/
    OR.2: /∨|\||(?i:or)\b/
    XOR.2: /⊕|(?i:xor)\b/
    IMPLIES.2: /→|->|=>|(?i:implies)\b/
    IFF.2: /↔|⟷|<->|<=>|(?i:iff)\b/
    EQUAL: "="
    DOT: "." | ":"
    NAME: /[?]?[\w’']+(?:\.[0-9][\w’']*)*/u
    %import common.WS
    %ignore WS
"""
LOGIC_PARSER = Lark(LOGIC_GRAMMAR, parser="lalr")


class Formula(Transformer):
    def names(self, items):
        return [str(item) for item in items]

    terms = names

    def predicate(self, items):
        return [str(items[0]), *(items[1] if len(items) > 1 and items[1] else [])]

    def proposition(self, items):
        return [str(items[0])]

    def equality(self, items):
        return ["=", str(items[0]), str(items[-1])]

    def negation(self, items):
        return ["not", items[-1]]

    def quantified(self, items):
        kind = "forall" if str(items[0]).lower() in {"∀", "forall"} else "exists"
        tree = items[-1]
        for name in reversed(items[1]):
            tree = [kind, name, tree]
        return tree

    def conjunction(self, items):
        values = items[::2]
        return values[0] if len(values) == 1 else ["and", *values]

    def disjunction(self, items):
        values = items[::2]
        return values[0] if len(values) == 1 else ["or", *values]

    def exclusive(self, items):
        values = items[::2]
        tree = values[0]
        for value in values[1:]:
            tree = ["xor", tree, value]
        return tree

    def implication(self, items):
        return items[0] if len(items) == 1 else ["implies", items[0], items[-1]]

    def equivalence(self, items):
        values = items[::2]
        tree = values[0]
        for value in values[1:]:
            tree = ["iff", tree, value]
        return tree


def formula_tree(value):
    if isinstance(value, list):
        return value
    tree = Formula().transform(LOGIC_PARSER.parse(value.strip().rstrip('.')))

    def variables(node, bound):
        op, *args = node
        if op in {"forall", "exists"}:
            name, body = args
            variable = name if name.startswith("?") else "?" + name
            return [op, variable, variables(body, {**bound, name: variable})]
        if op in {"not", "and", "or", "xor", "implies", "iff"}:
            return [op, *(variables(arg, bound) for arg in args)]
        return [op, *(bound.get(arg, "?" + arg if arg in {"x", "y", "z"} else arg)
                      for arg in args)]

    return variables(tree, {})


def direct_logic(context, query):
    if isinstance(context, str):
        context = [part.strip() for part in re.split(r"[;\n]+|(?<=\))\.\s+", context) if part.strip()]
    if not isinstance(context, list):
        raise ValueError("Logic premises must be a list or separated by newlines")
    return json.dumps({"premises": [formula_tree(p) for p in context], "query": formula_tree(query)})

## Inputs

In [11]:
import ast
import json
import re
import time
import pandas as pd
import torch
from lark.exceptions import LarkError
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def prepare_data(data):
    data = data.reset_index(drop=True).copy()
    if "id" not in data:
        data["id"] = next((data[c] for c in ["question_id", "example_id"] if c in data), data.index)
    if data.id.isna().any() or data.id.duplicated().any():
        raise ValueError("IDs must be present and unique")
    rows = []
    for row in data.to_dict("records"):
        logic = row.get("logic")
        if isinstance(logic, str):
            logic = json.loads(logic)
        if not isinstance(logic, dict):
            logic = None
        fol_context = next((row[c] for c in ["premises-FOL", "premises_fol"]
                            if isinstance(row.get(c), (str, list))), None)
        fol_query = next((row[c] for c in ["conclusion-FOL", "conclusion_fol", "query-FOL", "query_fol"]
                          if isinstance(row.get(c), (str, list))), None)
        if logic is not None:
            fol_context, fol_query = logic["premises"], logic["query"]
        context = next((row[c] for c in ["context", "premises", "given_info"]
                        if isinstance(row.get(c), (str, list))), None)
        statement = next((row[c] for c in ["statement", "conclusion", "question", "query"]
                          if isinstance(row.get(c), (str, list))), None)
        direct = fol_context is not None and fol_query is not None
        if direct:
            context, statement = fol_context, fol_query
        if isinstance(context, str) and context.lstrip().startswith("["):
            try:
                context = ast.literal_eval(context)
            except (ValueError, SyntaxError):
                pass
        if isinstance(statement, str) and statement.lstrip().startswith("["):
            try:
                statement = ast.literal_eval(statement)
            except (ValueError, SyntaxError):
                pass
        symbolic = r"[∀∃¬∧∨→↔⊕]|<->|->|=>|[&|~]|\b(?:forall|exists)\b|\b\w+\s*\("
        direct = direct or str(row.get("input_format", "")).casefold() == "logic"
        direct = direct or isinstance(statement, list) or (
            isinstance(statement, str) and bool(re.search(symbolic, statement, re.I)))
        direct = direct or isinstance(context, list) and any(isinstance(s, list) for s in context)
        if not direct and isinstance(statement, str) and re.search(symbolic, str(context), re.I):
            try:
                formula_tree(statement)
                direct = True
            except LarkError:
                pass
        if direct and isinstance(context, str):
            context = [part.strip() for part in re.split(r"[;\n]+|(?<=\))\.\s+", context) if part.strip()]
        if not isinstance(context, (str, list)):
            raise ValueError(f"Invalid context for {row['id']}")
        if isinstance(context, list):
            if not all(isinstance(s, str) or direct and isinstance(s, list) for s in context):
                raise ValueError(f"Invalid premises for {row['id']}")
            context = [s.strip() if isinstance(s, str) else s for s in context
                       if s and (not isinstance(s, str) or s.strip())]
        if not (isinstance(statement, str) and statement.strip() or direct and isinstance(statement, list) and statement):
            raise ValueError(f"Invalid statement for {row['id']}")
        mode = row.get("mode")
        if mode is None or pd.isna(mode) or mode == "auto":
            mode = {"ruletaker": "cwa", "counterbench": "causal"}.get(str(row.get("dataset", "")).casefold())
            text = context if isinstance(context, str) else " ".join(str(s) for s in context)
            if mode is None:
                mode = "causal" if not direct and re.search(r"\b(causes?|caused|leads? to)\b", text, re.I) and re.search(r"\b(would|counterfactual|intervention)\b", statement, re.I) else "classical"
        if mode not in {"classical", "cwa", "causal"}:
            raise ValueError(f"Unknown mode: {mode}")
        rows.append({"id": row["id"], "context": context, "statement": statement, "mode": mode, "direct": direct})
    if not rows:
        raise ValueError("The dataset is empty")
    return rows

## Prediction

In [12]:
def predict(data, repo):
    rows = prepare_data(data)
    results, pending = {}, []
    model_seconds = 0.0
    for row in rows:
        begin = time.perf_counter()
        if row["direct"]:
            try:
                text = direct_logic(row["context"], row["statement"])
                row["parser_seconds"] = time.perf_counter() - begin
                result = score(row, text)
            except (LarkError, ValueError, TypeError, IndexError) as e:
                row["parser_seconds"] = time.perf_counter() - begin
                result = {**row, "flag": False, "status": "parse_error", "translation": None,
                          "error": str(e), "reasoning_seconds": 0.0}
            results[row["id"]] = {**result, "parser": "logic", "translation_seconds": 0.0}
            continue
        text = parse_controlled(row["context"], row["statement"], row["mode"])
        row["parser_seconds"] = time.perf_counter() - begin
        result = score(row, text) if text is not None else None
        if result is not None and result["status"] != "parse_error":
            results[row["id"]] = {**result, "parser": "grammar",
                                  "translation_seconds": 0.0}
        else:
            pending.append(row)
    if pending:
        torch.manual_seed(42)
        begin = time.perf_counter()
        model, tokenizer = load_model(repo)
        model_seconds = time.perf_counter() - begin
        for mode in ["classical", "cwa", "causal"]:
            selected = [r for r in pending if r["mode"] == mode]
            for start in range(0, len(selected), 8):
                batch = selected[start:start + 8]
                if model.device.type == "cuda":
                    torch.cuda.synchronize(model.device)
                begin = time.perf_counter()
                texts = translate(batch, model, tokenizer, mode)
                if model.device.type == "cuda":
                    torch.cuda.synchronize(model.device)
                elapsed = time.perf_counter() - begin
                for row, text in zip(batch, texts):
                    results[row["id"]] = {**score(row, text), "parser": "model",
                                          "translation_seconds": elapsed / len(batch)}
        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    output = pd.DataFrame([results[row["id"]] for row in rows])
    output.attrs["model_load_seconds"] = model_seconds
    output.attrs["model"] = repo
    output.attrs["revision"] = MODELS[repo]
    return output

## Metrics

In [13]:
def evaluation_metrics(data, results, elapsed):
    metrics = {"examples": len(results), "total_seconds": elapsed,
               "seconds_per_query": elapsed / len(results), "queries_per_second": len(results) / elapsed,
               "model_load_seconds": results.attrs["model_load_seconds"],
               "grammar_seconds": results.parser_seconds.sum(),
               "translation_seconds": results.translation_seconds.sum(),
               "reasoning_seconds": results.reasoning_seconds.sum(),
               "parse_errors": results.status.eq("parse_error").sum(),
               "timeouts": results.status.eq("timeout").sum()}
    values = pd.Series(index=data.index, dtype="object")
    for column in ["label", "gold", "answer", "flag"]:
        if column in data:
            values = values.combine_first(data[column])
    report, matrix = pd.DataFrame(), pd.DataFrame()
    metrics["accuracy"] = float("nan")
    metrics["labeled_examples"] = 0
    if values.notna().any():
        labels = {"true": True, "yes": True, "1": True, "1.0": True,
                  "entailed": True, "entailment": True, "false": False, "no": False,
                  "0": False, "0.0": False, "refuted": False, "not entailment": False,
                  "unknown": False, "uncertain": False}
        values = values.reset_index(drop=True)
        mask = values.notna()
        if mask.any():
            text = values[mask].astype(str).str.strip().str.casefold()
            if not text.isin(labels).all():
                raise ValueError(f"Unrecognized labels: {text[~text.isin(labels)].unique().tolist()}")
            truth, prediction = text.map(labels), results.loc[mask, "flag"]
            metrics["labeled_examples"] = int(mask.sum())
            metrics["accuracy"] = accuracy_score(truth, prediction)
            report = pd.DataFrame(classification_report(truth, prediction, labels=[False, True],
                                  target_names=["False", "True"], output_dict=True, zero_division=0)).T
            matrix = pd.DataFrame(confusion_matrix(truth, prediction, labels=[False, True]),
                                  index=["Actual False", "Actual True"],
                                  columns=["Predicted False", "Predicted True"])
    return pd.Series(metrics, name="Evaluation", dtype=object), report, matrix

## Public data

In [14]:
import hashlib
import urllib.request

DATA_SOURCES = {
    "ruletaker/train-00000-of-00001-52adaa842dd7ed92.parquet": "https://huggingface.co/datasets/tasksource/ruletaker/resolve/a3e0880baeb6ec3d478f4c4d85afe04b21b6cf7f/data/train-00000-of-00001-52adaa842dd7ed92.parquet",
    "ruletaker/dev-00000-of-00001-480b21836eb3db96.parquet": "https://huggingface.co/datasets/tasksource/ruletaker/resolve/a3e0880baeb6ec3d478f4c4d85afe04b21b6cf7f/data/dev-00000-of-00001-480b21836eb3db96.parquet",
    "CounterBench/data_balanced_alpha_V1.json": "https://huggingface.co/datasets/CounterBench/CounterBench/resolve/c6225dfa00b7a89ccfeae67797b9a6d80e48e44c/data_balanced_alpha_V1.json",
    "CounterBench/data_balanced_backdoor_V2.json": "https://huggingface.co/datasets/CounterBench/CounterBench/resolve/c6225dfa00b7a89ccfeae67797b9a6d80e48e44c/data_balanced_backdoor_V2.json",
    "FOLIO/train.jsonl": "https://raw.githubusercontent.com/Yale-LILY/FOLIO/5d7bb84c7edab3fb358e057d2807f19cf5cf5e2d/data/v0.0/folio-train.jsonl",
    "FOLIO/validation.jsonl": "https://huggingface.co/datasets/benlipkin/folio/resolve/93480a9c4b3f2adc01b403d1447411b294e52253/validation.jsonl",
}


def download_data():
    root = Path("data/datasets/ai-assignment")
    for name, url in DATA_SOURCES.items():
        path = root / name
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, path)
    return root


def counter_group(row):
    names = {}
    return re.sub(r"\b[A-Z][a-z]+\b", lambda m: m[0] if m[0] in {"We", "Bern"}
                  else names.setdefault(m[0], "v" + str(len(names))), row["given_info"])


def datasets(split):
    root = download_data()
    rt = pd.read_parquet(next((root / "ruletaker").glob("train*" if split == "train" else "dev*")))
    rt = rt.groupby("config", group_keys=False).sample(3 if split == "train" else 16, random_state=42)
    rows = [{"id": f"ruletaker:{i}", "dataset": "RuleTaker", "mode": "cwa",
             "context": r.context, "statement": r.question,
             "gold": "entailed" if r.label == "entailment" else "unknown"} for i, r in rt.iterrows()]
    folio = pd.read_json(root / "FOLIO" / ("train.jsonl" if split == "train" else "validation.jsonl"), lines=True)
    if split == "train":
        folio = folio.groupby("label", group_keys=False).sample(16, random_state=42)
    for i, r in folio.iterrows():
        rows.append({"id": f"folio:{i}", "dataset": "FOLIO", "mode": "classical",
                     "context": r["premises"], "statement": r["conclusion"],
                     "gold": {"True": "entailed", "False": "refuted", "Unknown": "unknown", "Uncertain": "unknown"}[r.label]})
    counter = []
    for path in sorted((root / "CounterBench").glob("data*.json")):
        for r in json.loads(path.read_text()):
            holdout = int(hashlib.sha256(counter_group(r).encode()).hexdigest()[:8], 16) % 5 == 0
            if holdout == (split != "train"):
                counter.append({**r, "file": path.stem})
    counter = pd.DataFrame(counter)
    if split == "train":
        counter = counter.groupby("type", group_keys=False).sample(4, random_state=42)
    for _, r in counter.iterrows():
        rows.append({"id": f"counterbench:{r['file']}:{r.question_id}", "dataset": "CounterBench", "mode": "causal",
                     "context": r.given_info, "statement": r.question,
                     "gold": "entailed" if r.answer == "yes" else "unknown"})
    return rows

## Evaluation

In [15]:
data = None
# data = pd.read_csv("data/datasets/private.csv", dtype={"id": str})
# data = pd.read_json("data/datasets/private.jsonl", lines=True)
if data is None:
    data = pd.DataFrame(datasets("validation"))

started = time.perf_counter()
predictions = predict(data, MODEL_ID)
metrics, report, matrix = evaluation_metrics(data, predictions, time.perf_counter() - started)

Path("submission").mkdir(exist_ok=True)
predictions[["id", "flag"]].to_csv("submission/submission.csv", index=False)
output = Path("data/outputs/ai-assignment-evaluation")
output.mkdir(parents=True, exist_ok=True)
predictions.to_json(output / "predictions.jsonl", orient="records", lines=True)
metrics.to_json(output / "metrics.json")

display(metrics.to_frame())
if not report.empty:
    display(report, matrix)
display(predictions.status.value_counts().rename("count").to_frame())

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

,Evaluation
examples,532
total_seconds,125.19854
seconds_per_query,0.235336
queries_per_second,4.249251
model_load_seconds,1.656316
grammar_seconds,1.313691
translation_seconds,120.964779
reasoning_seconds,1.223036
parse_errors,38
timeouts,0


,precision,recall,f1-score,support
False,0.872222,0.978193,0.922173,321.000000
True,0.959302,0.781991,0.861619,211.000000
accuracy,0.900376,0.900376,0.900376,0.900376
macro avg,0.915762,0.880092,0.891896,532.000000
weighted avg,0.906760,0.900376,0.898156,532.000000


,Predicted False,Predicted True
Actual False,314,7
Actual True,46,165


,count
status,
refuted,137
fixed_point,128
unknown,106
entailed,97
parse_error,38
inconsistent_evidence,20
inconsistent,6
